In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats
import pickle
import json

In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
ctl = 'goamazon_2pulse.smalldom.r20251008.rerun'
ehe1 = 'goamazon_2pulse.smalldom.ehe1.r20251030.rerun'
csd_stats = {}
with open(f'{ctl}/pkl/csd_stats.pkl', 'rb') as f:
    csd_stats[ctl] = pickle.load(f)
with open(f'{ehe1}/pkl/csd_stats.pkl', 'rb') as f:
    csd_stats[ehe1] = pickle.load(f)

In [ ]:
minmf = 0
print(np.max(csd_stats[ehe1][0]))
print(np.max(csd_stats[ctl][0]))
maxmf = np.max([np.max(csd_stats[ehe1][0]), np.max(csd_stats[ctl][0])]) + 10.0
# maxmf = np.max(csd_stats[ctl][0]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)

In [ ]:
nx, ny, nz, nt = 512, 512, 100, 241
dts = 0.5 # minute
dx = 50 # m
dy = 50 # m
dz = 50 # m
grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3
z = np.arange(dz/2, 5000., dz)
t = np.arange(0, 241)*dts # minutes
ti = np.arange(-0.5, 241., 1.)*dts
zi = np.arange(0., 5001., dz)

In [ ]:
def binned_wq(csd_stats, casename, bins):
    global nx, ny, nz, nt
    print(nx, ny, nz, nt)
    with open(f'{casename}/pkl/plume_all_wq.pkl', 'rb') as f:
        wq = pickle.load(f)
    clipped_mf = csd_stats[0] 
    attached_ind = csd_stats[-1]
    bin_ids = np.digitize(np.log10(clipped_mf), bins)
    nbins = len(bins) - 1
    factor = 1.0/float(nx*ny*nt)
    sum_wq = np.zeros((nbins, nz))
    mean_wq = np.zeros((nbins, nz))
    total_count = 0
    for bnm in range(nbins):
        bn = bnm + 1
        cn = attached_ind[bin_ids==bn]
        total_count += len(cn)
        print(bn, len(cn))
        if len(cn) > 0:
            sum_wq[bnm, :] = (wq[cn, :, :].sum(axis=1)*factor).sum(axis=0)
            mean_wq[bnm, :] = (wq[cn, :, :].sum(axis=1)*factor).mean(axis=0)
    return sum_wq, mean_wq 

In [ ]:
# minmf = 0
# print(np.max(csd_stats[ctl][0]))
# print(np.max(csd_stats[ehe21][0]))
# maxmf = np.max([np.max(csd_stats[ctl][0]), np.max(csd_stats[ehe21][0])]) + 10.0
# maxmf = np.log10(maxmf)
print(minmf, maxmf)
bins = np.linspace(minmf, maxmf, 11)

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
n, bins, patches = ax.hist([np.log10(csd_stats[ehe1][0]), np.log10(csd_stats[ctl][0])], bins=bins, log=True, color=['red', 'black'])
ax.axvline(np.median(np.log10(csd_stats[ehe1][0])), color='red', linestyle='--')
ax.axvline(np.median(np.log10(csd_stats[ctl][0])), color='black', linestyle='--')
ax.legend(['EHE1', 'CTL'])
ax.set_ylim(1, 3.0e4)
ax.set_xlabel(f'Mean cloud-base mass flux (kg/s)')
plt.show()

In [ ]:
sum_wq = {}
mean_wq = {}
sum_wq[ctl], mean_wq[ctl] = binned_wq(csd_stats[ctl], ctl, bins)
sum_wq[ehe1], mean_wq[ehe1] = binned_wq(csd_stats[ehe1], ehe1, bins)

In [ ]:
fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))
# ax.plot(total_qcl_diff[-1,:]*1.0e3, z*1.0e-3, color='black', linewidth=2.0, marker='+', label="total diff")
ax.plot((sum_wq[ehe1].sum(axis=0) - sum_wq[ctl].sum(axis=0))*1.0e3, z*1.0e-3, color='black', linewidth=2.0, label="total diff by summing")
ax.set_ylim((0, 5))
ax.set_ylabel('Height (km)')
plt.legend(loc='upper left', fontsize=14)
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(12, 16))
axs = axs.flatten()

for i in range(4, 10):
    ax = axs[i-4]
    ax.plot(mean_wq[ctl][i,:]*1e3, z*1.0e-3, color='black', linewidth=2.5, label='CTL')
    ax.plot(mean_wq[ehe1][i,:]*1e3, z*1.0e-3, color='red', linewidth=2.5, label='EHE1')
    ax.set_ylim((0, 4.5))
    ax.set_xlim()
    ax.grid(alpha=0.3)
    
    # Add labels only to left column and bottom row
    if (i-4) % 3 == 0:
        ax.set_ylabel('Height (km)')
    if (i-4) >= 3:
        ax.set_xlabel(r"Mean $\overline{w'q'}$ (g kg$^{-1}$ s$^{-1}$)", fontsize=16)
    
    # Simpler title with bin range
    ax.set_title(fr'Bin {i}: {10**(bins[i]):.0f}$\minus${10**(bins[i+1]):.0f} kg/s', fontsize=14)
    
    # Legend only in first subplot
    if i == 4:
        ax.legend(loc='upper right', fontsize=11, frameon=True, framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
import copy
diff2d = sum_wq[ehe1] - sum_wq[ctl]
diff2d = diff2d*1.0e3
print(diff2d.max(), diff2d.min())
fig, axs = plt.subplots(1, 1, figsize=(12, 12))
levels = np.concatenate([np.arange(-10.0, -0.01, 1.0), [-0.01, 0.01], np.arange(1.0, 10.01, 1.0)])
cmap = copy.deepcopy(mpl.cm.bwr)
norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
cm = axs.pcolormesh(bins, zi, diff2d.T, norm=norm, cmap=cmap, shading='flat')
axs.set_ylim(0, 4000)
plt.colorbar(cm)
plt.show()